Proceso de carga de datos.

In [ ]:
import os
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report
from contextlib import redirect_stdout
from TheModel import build
import io

# Ruta a la carpeta que contiene los modelos guardados
carpeta_modelos = './modelos/'

# Función para cargar modelos

def cargar_modelos(ruta):
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"La carpeta '{ruta}' no existe.")

    modelos = [
        tf.keras.models.load_model(os.path.join(ruta, archivo))
        for archivo in os.listdir(ruta)
        if archivo.endswith('.keras')
    ]

    if not modelos:
        raise ValueError("No se cargó ningún modelo. Asegúrate de que existan archivos .keras en la carpeta.")

    print(f"✅ Se han cargado {len(modelos)} modelos exitosamente desde '{ruta}'.")
    return modelos

loaded_local_models = cargar_modelos(carpeta_modelos)


C:\Users\hendr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\keras\src\layers\layer.py:361: UserWarning: `build()` was called on layer 'attention_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
C:\Users\hendr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\keras\src\layers\layer.py:361: UserWarning: `build()` was called on layer 'attention_block_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to 

✅ Se han cargado 5 modelos exitosamente desde './modelos/'.


In [46]:

# Cargar datos de test
datos_test = np.load("local_data_splits/datos_test_mnist.npz")
x_test, y_test = datos_test["x_test"], datos_test["y_test"]

modelos que se utilizarán para el aprendizaje federado.

In [47]:
def fed_avg(models):
    weights = [model.get_weights() for model in models]
    return [np.mean(layer_weights, axis=0) for layer_weights in zip(*weights)]


In [48]:
def fed_trimmed_mean(models, trim_ratio=0.2):
    weights = [model.get_weights() for model in models]
    trimmed_weights = []
    for layer_weights in zip(*weights):
        sorted_layer = np.sort(layer_weights, axis=0)
        trim = int(trim_ratio * len(sorted_layer))
        trimmed_layer = sorted_layer[trim:-trim] if trim > 0 else sorted_layer
        trimmed_weights.append(np.mean(trimmed_layer, axis=0))
    return trimmed_weights

In [49]:
def fed_median(models):
    weights = [model.get_weights() for model in models]
    return [np.median(layer_weights, axis=0) for layer_weights in zip(*weights)]


verifica que todas la arquitecturas de los modelos son iguales.

In [50]:
# Verificar la arquitectura de los modelos
def get_model_summary_str(model):
    summary_io = io.StringIO()
    with redirect_stdout(summary_io):
        model.summary()
    return summary_io.getvalue()

def verificar_arquitectura(modelos):
    for i in range(len(modelos) - 1):
        assert get_model_summary_str(modelos[i]) == get_model_summary_str(modelos[i+1]), \
            "⚠️ Models have different architectures"
    print("✅ Todos los modelos tienen la misma arquitectura según summary().")
    

verificar_arquitectura(loaded_local_models)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 28, 28)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 28, 32)         │         5,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_block_1               │ (None, 28, 32)         │         1,144 │
│ (AttentionBlock)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,384 (91.35 KB)

 Trainable params: 7,794 (30.45 KB)

 Non-trainable params: 0 (0.00 B)

...

 Total params: 23,384 (91.35 KB)

 Trainable params: 7,794 (30.45 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 15,590 (60.90 KB)

✅ Todos los modelos tienen la misma arquitectura según summary().


Creación, evaluación y guardado de los modelos globales.

In [51]:

# Crear, evaluar y guardar modelos globales
def create_global_model(aggregated_weights, save_name, input_shape):
    global_model = build.build_it(input_shape=input_shape)
    global_model.set_weights(aggregated_weights)
    y_pred = global_model.predict(x_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    print(f"\nReporte para modelo {save_name}:")
    print(classification_report(y_test, y_pred_classes))
    global_model.save(save_name)
    print(f"✅ Modelo global guardado como {save_name}")

# Ejecutar y guardar modelos agregados
input_shape = loaded_local_models[0].input_shape[1:]

create_global_model(fed_avg(loaded_local_models), "global_model_avg.keras", input_shape)
create_global_model(fed_median(loaded_local_models), "global_model_median.keras", input_shape)
create_global_model(fed_trimmed_mean(loaded_local_models), "global_model_trimmed.keras", input_shape)


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step

Reporte para modelo global_model_avg.keras:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       980
           1       0.51      0.19      0.27      1135
           2       0.00      0.00      0.00      1032
           3       0.03      0.01      0.01      1010
           4       0.00      0.00      0.00       982
           5       0.00      0.00      0.00       892
           6       0.00      0.00      0.00       958
           7       0.00      0.00      0.00      1028
           8       0.10      0.98      0.19       974
           9       0.00      0.00      0.00      1009

    accuracy                           0.12     10000
   macro avg       0.06      0.12      0.05     10000
weighted avg       0.07      0.12      0.05     10000

✅ Modelo global guardado como global_model_avg.keras


C:\Users\hendr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\hendr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\hendr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classificati

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step

Reporte para modelo global_model_median.keras:
              precision    recall  f1-score   support

           0       0.25      0.08      0.12       980
           1       0.19      0.36      0.25      1135
           2       0.00      0.00      0.00      1032
           3       0.13      0.98      0.23      1010
           4       0.00      0.00      0.00       982
           5       0.06      0.01      0.01       892
           6       0.00      0.00      0.00       958
           7       0.00      0.00      0.00      1028
           8       0.00      0.00      0.00       974
           9       0.00      0.00      0.00      1009

    accuracy                           0.15     10000
   macro avg       0.06      0.14      0.06     10000
weighted avg       0.06      0.15      0.06     10000

✅ Modelo global guardado como global_model_median.keras


C:\Users\hendr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\hendr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\hendr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classificati

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step

Reporte para modelo global_model_trimmed.keras:
              precision    recall  f1-score   support

           0       0.09      0.02      0.03       980
           1       0.24      0.09      0.13      1135
           2       0.00      0.00      0.00      1032
           3       0.13      0.44      0.20      1010
           4       0.00      0.00      0.00       982
           5       0.00      0.00      0.00       892
           6       0.00      0.00      0.00       958
           7       0.00      0.00      0.00      1028
           8       0.09      0.52      0.15       974
           9       0.00      0.00      0.00      1009

    accuracy                           0.11     10000
   macro avg       0.05      0.11      0.05     10000
weighted avg       0.06      0.11      0.05     10000

✅ Modelo global guardado como global_model_trimmed.keras


C:\Users\hendr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\hendr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\hendr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classificati